# Adaptive FSM evaluation for the saved Robot SLM

This ready-to-run notebook evaluates the adaptive FSM/cache against the ordered 240-row Drive evaluation stream. It does four things:

1. Load the already-trained Robot SLM bundle.
2. Try trusted deterministic FSM/FST patterns first.
3. Send uncertain requests to the SLM and keep confirmed recurring patterns in a decayed candidate cache.
4. Promote sufficiently repeated and reliable candidates into the FSM.

The existing eight-field `Decision`, validator, dialogue memory, and authoritative inventory remain unchanged. Drawer locations and final spoken responses are never learned or cached.


In [ ]:
#@title Install the saved SLM runtime dependencies
%pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.48.1 'pydantic>=2.10,<3' 'sentencepiece>=0.2' 'pandas>=2.0'


In [ ]:
#@title Mount Drive and choose the saved bundle
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import glob

# Leave blank to use the newest saved robot_slm_bundle.
SLM_BUNDLE_DIR = "" #@param {type:"string"}
DATASET_PATH = "/content/drive/MyDrive/Robot SLM Supplemental Datasets/Adaptive FSM Evaluation v1/robot_slm_adaptive_fsm_evaluation_v1.jsonl" #@param {type:"string"}
MANIFEST_PATH = "/content/drive/MyDrive/Robot SLM Supplemental Datasets/Adaptive FSM Evaluation v1/robot_slm_adaptive_fsm_evaluation_v1_manifest.json" #@param {type:"string"}
STREAM_SPLIT = "test" #@param ["train", "validation", "test"]
MAX_STREAM_ROWS = 0 #@param {type:"integer"}

# Candidate-cache and routing thresholds.
FSM_USE_THRESHOLD = 0.92 #@param {type:"number"}
CACHE_MERGE_THRESHOLD = 0.75 #@param {type:"number"}
MIN_TOP2_MARGIN = 0.05 #@param {type:"number"}
HEAT_DECAY = 0.995 #@param {type:"number"}
PROMOTION_MIN_CONFIRMED = 2 #@param {type:"integer"}
PROMOTION_MIN_SHADOW = 1 #@param {type:"integer"}
PROMOTION_MIN_EXAMPLES = 2 #@param {type:"integer"}
PROMOTION_MIN_VARIANTS = 2 #@param {type:"integer"}
PROMOTION_MIN_ACCURACY = 1.0 #@param {type:"number"}
PROMOTION_MIN_HEAT = 1.5 #@param {type:"number"}

if SLM_BUNDLE_DIR:
    BUNDLE_DIR = Path(SLM_BUNDLE_DIR)
else:
    found = sorted(
        Path(path) for path in glob.glob(
            '/content/drive/MyDrive/robot_slm_runs/*/robot_slm_bundle'
        ) if (Path(path) / 'runtime_config.json').is_file()
    )
    if not found:
        raise FileNotFoundError('No saved robot_slm_bundle was found. Run the SLM notebook first or set SLM_BUNDLE_DIR.')
    BUNDLE_DIR = found[-1]

print('SLM bundle:', BUNDLE_DIR)
print('Dataset:', DATASET_PATH)
print('Manifest:', MANIFEST_PATH)


In [ ]:
#@title Verify and load the supplied behavior dataset
import hashlib
import json
from collections import Counter

dataset_path = Path(DATASET_PATH)
manifest_path = Path(MANIFEST_PATH)
if not dataset_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError('The dataset or manifest path is incorrect.')

manifest = json.loads(manifest_path.read_text())
actual_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
if actual_sha256 != manifest['sha256']:
    raise RuntimeError('Dataset SHA-256 does not match the supplied manifest.')

rows = [json.loads(line) for line in dataset_path.read_text().splitlines() if line.strip()]
if len(rows) != manifest['audit']['rows']:
    raise RuntimeError('Dataset row count does not match the supplied manifest.')

stream_rows = [row for row in rows if row['split'] == STREAM_SPLIT]
if MAX_STREAM_ROWS > 0:
    stream_rows = stream_rows[:MAX_STREAM_ROWS]

print('Verified SHA-256:', actual_sha256)
print('All rows:', len(rows), '| stream rows:', len(stream_rows), '| split:', STREAM_SPLIT)
print('Stream scenarios:', dict(Counter(row['scenario'] for row in stream_rows)))


In [ ]:
#@title Load the saved SLM and its exact runtime contract
import importlib.util
import sys

runtime_path = BUNDLE_DIR / 'slm_runtime.py'
if not runtime_path.is_file():
    raise FileNotFoundError(f'Missing saved runtime: {runtime_path}')

spec = importlib.util.spec_from_file_location('saved_robot_slm_runtime', runtime_path)
Runtime = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = Runtime
spec.loader.exec_module(Runtime)
engine = Runtime.load_runtime(BUNDLE_DIR)
bundle_inventory = json.loads((BUNDLE_DIR / 'inventory.json').read_text())

print('Loaded SLM bundle:', BUNDLE_DIR)
print('Decision fields:', list(Runtime.Decision.model_fields))
print('Bundle inventory items:', len(bundle_inventory))


## Adaptive FSM and candidate cache

A candidate cache entry contains a state-conditioned language pattern, a parameterized `Decision` template, examples, heat, and confirmation statistics. Unpromoted candidates never answer requests. A promoted rule still has to bind its slots against the current inventory and pass the saved SLM runtime validator.


In [ ]:
from __future__ import annotations

import copy
import math
import re
import time
from dataclasses import dataclass, field
from difflib import SequenceMatcher

# These words may surround a direct item request without changing its meaning.
FILLER_WORDS = {
    'a', 'an', 'the', 'some', 'please', 'pls', 'i', 'me', 'my', 'want',
    'need', 'would', 'like', 'can', 'could', 'you', 'do', 'have', 'get',
    'give', 'show', 'find', 'locate', 'where', 'is', 'are', 'tell'
}
NEGATION_OR_CORRECTION = {
    'no', 'not', 'dont', 'didnt', 'isnt', 'without', 'except', 'instead', 'wrong'
}
CACHEABLE_ACTIONS = {'locate', 'clarify', 'not_found'}


def words(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())


def latest_user_text(payload):
    for turn in reversed(payload.get('conversation', [])):
        if turn.get('role') == 'user':
            return turn.get('text', '')
    return ''


def state_signature(payload):
    memory = payload.get('memory') or {}
    compact = {
        'decision_mode': payload.get('decision_mode'),
        'state': payload.get('state'),
        'has_pending_family': bool(memory.get('pending_item_family')),
        'known_attribute_keys': sorted((memory.get('known_attributes') or {}).keys()),
        'missing_attributes': sorted(memory.get('missing_attributes') or []),
        'last_robot_action': memory.get('last_robot_action'),
        'has_proposed_answer': payload.get('proposed_robot_answer') is not None,
    }
    return json.dumps(compact, sort_keys=True, separators=(',', ':'))


def _lexicon(inventory):
    entries = {}
    for item in inventory:
        for phrase in {item['name'], *item.get('aliases', [])}:
            token_key = tuple(words(phrase))
            if token_key:
                entries.setdefault(token_key, []).append(('item', item['id'], item))
        family_phrase = item['family'].replace('_', ' ')
        token_key = tuple(words(family_phrase))
        if token_key:
            entries.setdefault(token_key, []).append(('family', item['family'], None))
        for key, value in item.get('attributes', {}).items():
            token_key = tuple(words(value))
            if token_key:
                entries.setdefault(token_key, []).append(('attribute', (key, value), None))
    return entries


def request_features(payload, inventory):
    text = latest_user_text(payload)
    token_list = words(text)
    entries = _lexicon(inventory)
    phrases = sorted(entries, key=lambda value: (-len(value), value))
    priority = {'item': 3, 'family': 2, 'attribute': 1}

    output = []
    slots = {'item_id': None, 'family': None, 'attributes': {}}
    ambiguous = False
    index = 0
    while index < len(token_list):
        matches = []
        for phrase in phrases:
            width = len(phrase)
            if token_list[index:index + width] == list(phrase):
                for descriptor in entries[phrase]:
                    matches.append((width, priority[descriptor[0]], descriptor))
        if not matches:
            if token_list[index] not in FILLER_WORDS:
                output.append(token_list[index])
            index += 1
            continue

        best_width = max(match[0] for match in matches)
        matches = [match for match in matches if match[0] == best_width]
        best_priority = max(match[1] for match in matches)
        descriptors = [match[2] for match in matches if match[1] == best_priority]
        kind = descriptors[0][0]

        if kind == 'item':
            item_ids = {descriptor[1] for descriptor in descriptors}
            if len(item_ids) != 1:
                ambiguous = True
                output.append('<ambiguous-item>')
            else:
                item_id = next(iter(item_ids))
                item = next(item for item in inventory if item['id'] == item_id)
                if slots['item_id'] not in {None, item_id}:
                    ambiguous = True
                slots['item_id'] = item_id
                slots['family'] = item['family']
                slots['attributes'].update(item.get('attributes', {}))
                output.append('<item>')
        elif kind == 'family':
            families = {descriptor[1] for descriptor in descriptors}
            if len(families) != 1:
                ambiguous = True
                output.append('<ambiguous-family>')
            else:
                family = next(iter(families))
                if slots['family'] not in {None, family}:
                    ambiguous = True
                slots['family'] = family
                output.append('<family>')
        else:
            values = {descriptor[1] for descriptor in descriptors}
            if len(values) != 1:
                ambiguous = True
                output.append('<ambiguous-attribute>')
            else:
                key, value = next(iter(values))
                previous = slots['attributes'].get(key)
                if previous not in {None, value}:
                    ambiguous = True
                slots['attributes'][key] = value
                output.append(f'<attr:{key}>')
        index += best_width

    surface = ' '.join(output) or '<empty>'
    return {
        'text': text,
        'raw_normalized': ' '.join(token_list),
        'surface': surface,
        'slots': slots,
        'ambiguous': ambiguous,
        'has_negation_or_correction': bool(set(token_list) & NEGATION_OR_CORRECTION),
        'variant': json.dumps(slots, sort_keys=True, separators=(',', ':')),
    }


def surface_similarity(left, right):
    left_tokens, right_tokens = left.split(), right.split()
    left_set, right_set = set(left_tokens), set(right_tokens)
    union = left_set | right_set
    jaccard = len(left_set & right_set) / len(union) if union else 1.0
    sequence = SequenceMatcher(None, left, right).ratio()
    return 0.55 * sequence + 0.45 * jaccard


def decisions_equal(left, right):
    if left is None or right is None:
        return False
    left_dict = left.model_dump() if hasattr(left, 'model_dump') else dict(left)
    right_dict = right.model_dump() if hasattr(right, 'model_dump') else dict(right)
    return left_dict == right_dict


def make_template(decision, payload, features, inventory):
    data = decision.model_dump() if hasattr(decision, 'model_dump') else dict(decision)
    if payload.get('decision_mode') != 'dialogue_decision':
        return None
    if data['action'] not in CACHEABLE_ACTIONS or data['failure_detected']:
        return None
    if data['action'] == 'clarify' and data['item_family'] is None:
        return None

    by_id = {item['id']: item for item in inventory}
    item_slot = by_id.get(features['slots']['item_id'])
    memory = payload.get('memory') or {}

    if item_slot and item_slot['family'] == data['item_family']:
        family_source = 'input_item'
        fixed_family = None
    elif features['slots']['family'] == data['item_family']:
        family_source = 'input_family'
        fixed_family = None
    elif memory.get('pending_item_family') == data['item_family']:
        family_source = 'pending_family'
        fixed_family = None
    elif data['item_family'] is None:
        family_source = 'none'
        fixed_family = None
    else:
        family_source = 'fixed'
        fixed_family = data['item_family']

    attribute_sources = {}
    for key, value in data['attributes'].items():
        if features['slots']['attributes'].get(key) == value:
            attribute_sources[key] = {'source': 'input_attribute', 'value': None}
        elif item_slot and item_slot.get('attributes', {}).get(key) == value:
            attribute_sources[key] = {'source': 'input_item', 'value': None}
        elif (memory.get('known_attributes') or {}).get(key) == value:
            attribute_sources[key] = {'source': 'memory', 'value': None}
        else:
            attribute_sources[key] = {'source': 'fixed', 'value': value}

    if data['resolved_item_id'] is None:
        item_source = 'none'
    elif features['slots']['item_id'] == data['resolved_item_id']:
        item_source = 'input_item'
    else:
        target_item = by_id.get(data['resolved_item_id'])
        if not target_item:
            return None
        matches = [item for item in inventory if item['family'] == data['item_family'] and
                   all(item.get('attributes', {}).get(key) == value
                       for key, value in data['attributes'].items())]
        if len(matches) != 1:
            return None
        item_source = 'resolve_unique'

    return {
        'action': data['action'],
        'family_source': family_source,
        'fixed_family': fixed_family,
        'item_source': item_source,
        'attribute_sources': attribute_sources,
        'missing_attributes': list(data['missing_attributes']),
        'failure_detected': False,
        'failure_type': 'none',
        'recovery_strategy': 'none',
    }


def template_signature(template):
    return json.dumps(template, sort_keys=True, separators=(',', ':'))


def instantiate_template(template, payload, features, inventory):
    if features['ambiguous'] or features['has_negation_or_correction']:
        return None
    by_id = {item['id']: item for item in inventory}
    input_item = by_id.get(features['slots']['item_id'])
    memory = payload.get('memory') or {}

    source = template['family_source']
    if source == 'input_item':
        family = input_item['family'] if input_item else None
    elif source == 'input_family':
        family = features['slots']['family']
    elif source == 'pending_family':
        family = memory.get('pending_item_family')
    elif source == 'fixed':
        family = template['fixed_family']
    else:
        family = None

    attributes = {}
    for key, specification in template['attribute_sources'].items():
        attr_source = specification['source']
        if attr_source == 'input_attribute':
            value = features['slots']['attributes'].get(key)
        elif attr_source == 'input_item':
            value = input_item.get('attributes', {}).get(key) if input_item else None
        elif attr_source == 'memory':
            value = (memory.get('known_attributes') or {}).get(key)
        else:
            value = specification['value']
        if value is None:
            return None
        attributes[key] = value

    action = template['action']
    resolved_item_id = None
    if action == 'locate':
        if template['item_source'] == 'input_item':
            item = input_item
        else:
            matches = [item for item in inventory if item['family'] == family and
                       all(item.get('attributes', {}).get(key) == value
                           for key, value in attributes.items())]
            item = matches[0] if len(matches) == 1 else None
        if item is None:
            return None
        family = item['family']
        resolved_item_id = item['id']
        attributes = dict(item.get('attributes', {}))
    elif action == 'clarify':
        matching_family = [item for item in inventory if item['family'] == family and
                           all(item.get('attributes', {}).get(key) == value
                               for key, value in attributes.items())]
        valid_keys = {key for item in matching_family for key in item.get('attributes', {})}
        if len(matching_family) < 2 or not set(template['missing_attributes']) <= valid_keys:
            return None
    elif action == 'not_found':
        matches = [item for item in inventory if item['family'] == family and
                   all(item.get('attributes', {}).get(key) == value
                       for key, value in attributes.items())]
        if matches:
            return None

    decision = Runtime.Decision(
        action=action,
        item_family=family,
        resolved_item_id=resolved_item_id,
        attributes=attributes,
        missing_attributes=list(template['missing_attributes']),
        failure_detected=template['failure_detected'],
        failure_type=template['failure_type'],
        recovery_strategy=template['recovery_strategy'],
    )
    return Runtime.validate_decision(decision, payload, inventory)


def exact_seed_rule(payload, features, inventory):
    # The only initial direct rule: an exact, non-negated inventory item request.
    if payload.get('decision_mode') != 'dialogue_decision':
        return None
    if payload.get('state') != 'awaiting_request':
        return None
    if features['ambiguous'] or features['has_negation_or_correction']:
        return None
    if features['surface'] != '<item>' or not features['slots']['item_id']:
        return None
    item = next((item for item in inventory if item['id'] == features['slots']['item_id']), None)
    if item is None:
        return None
    decision = Runtime.Decision(
        action='locate', item_family=item['family'], resolved_item_id=item['id'],
        attributes=dict(item.get('attributes', {})), missing_attributes=[],
        failure_detected=False, failure_type='none', recovery_strategy='none'
    )
    return Runtime.validate_decision(decision, payload, inventory)


@dataclass
class CandidatePattern:
    pattern_id: str
    state_key: str
    template: dict
    surfaces: set[str] = field(default_factory=set)
    examples: set[str] = field(default_factory=set)
    variants: set[str] = field(default_factory=set)
    confirmed: int = 0
    shadow_successes: int = 0
    shadow_failures: int = 0
    served_successes: int = 0
    served_failures: int = 0
    heat: float = 0.0
    last_step: int = 0
    status: str = 'candidate'

    def decay_to(self, step):
        elapsed = max(0, step - self.last_step)
        self.heat *= HEAT_DECAY ** elapsed
        self.last_step = step

    @property
    def shadow_trials(self):
        return self.shadow_successes + self.shadow_failures

    @property
    def shadow_accuracy(self):
        return self.shadow_successes / self.shadow_trials if self.shadow_trials else 0.0


class PatternCache:
    def __init__(self):
        self.patterns = []
        self.next_id = 1
        self.promotion_events = []

    def decay_all(self, step):
        for pattern in self.patterns:
            pattern.decay_to(step)

    def score(self, pattern, state_key, surface):
        if pattern.state_key != state_key:
            return 0.0
        return max((surface_similarity(surface, example)
                    for example in pattern.surfaces), default=0.0)

    def learn_confirmed(self, payload, inventory, decision, step):
        features = request_features(payload, inventory)
        template = make_template(decision, payload, features, inventory)
        if template is None or features['ambiguous'] or features['has_negation_or_correction']:
            return None

        state_key = state_signature(payload)
        signature = template_signature(template)
        compatible = [pattern for pattern in self.patterns
                      if pattern.state_key == state_key and
                      template_signature(pattern.template) == signature]
        scored = [(self.score(pattern, state_key, features['surface']), pattern)
                  for pattern in compatible]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        if scored and scored[0][0] >= CACHE_MERGE_THRESHOLD:
            pattern = scored[0][1]
        else:
            pattern = CandidatePattern(
                pattern_id=f'pattern_{self.next_id:04d}',
                state_key=state_key,
                template=template,
                last_step=step,
            )
            self.next_id += 1
            self.patterns.append(pattern)

        pattern.decay_to(step)
        pattern.surfaces.add(features['surface'])
        pattern.examples.add(features['raw_normalized'])
        pattern.variants.add(features['variant'])
        pattern.confirmed += 1
        pattern.heat += 1.0
        self.maybe_promote(pattern, step)
        return pattern.pattern_id

    def shadow_test(self, payload, inventory, target, step):
        features = request_features(payload, inventory)
        state_key = state_signature(payload)
        for pattern in self.patterns:
            if pattern.status != 'candidate':
                continue
            score = self.score(pattern, state_key, features['surface'])
            if score < FSM_USE_THRESHOLD:
                continue
            try:
                proposed = instantiate_template(pattern.template, payload, features, inventory)
            except Exception:
                proposed = None
            if proposed is None:
                continue
            if decisions_equal(proposed, target):
                pattern.shadow_successes += 1
            else:
                pattern.shadow_failures += 1
            self.maybe_promote(pattern, step)

    def maybe_promote(self, pattern, step):
        pattern.decay_to(step)
        ready = (
            pattern.status == 'candidate' and
            pattern.confirmed >= PROMOTION_MIN_CONFIRMED and
            pattern.shadow_trials >= PROMOTION_MIN_SHADOW and
            len(pattern.examples) >= PROMOTION_MIN_EXAMPLES and
            len(pattern.variants) >= PROMOTION_MIN_VARIANTS and
            pattern.shadow_accuracy >= PROMOTION_MIN_ACCURACY and
            pattern.heat >= PROMOTION_MIN_HEAT
        )
        if ready:
            pattern.status = 'promoted'
            self.promotion_events.append({'step': step, 'pattern_id': pattern.pattern_id})

    def demote(self, pattern, step):
        pattern.status = 'candidate'
        pattern.served_failures += 1
        pattern.heat *= 0.5
        pattern.last_step = step


class AdaptiveRouter:
    def __init__(self, cache):
        self.cache = cache

    def try_fsm(self, payload, inventory, step):
        self.cache.decay_all(step)
        features = request_features(payload, inventory)
        try:
            seed = exact_seed_rule(payload, features, inventory)
        except Exception:
            seed = None
        if seed is not None:
            return seed, {'source': 'fsm_seed_exact_item', 'pattern_id': None, 'match_score': 1.0}

        if features['ambiguous'] or features['has_negation_or_correction']:
            return None, {'source': 'slm', 'reason': 'ambiguous_or_corrective_language'}

        state_key = state_signature(payload)
        scored = []
        for pattern in self.cache.patterns:
            if pattern.status != 'promoted':
                continue
            score = self.cache.score(pattern, state_key, features['surface'])
            if score > 0:
                scored.append((score, pattern))
        scored.sort(key=lambda pair: pair[0], reverse=True)
        if not scored or scored[0][0] < FSM_USE_THRESHOLD:
            return None, {'source': 'slm', 'reason': 'no_promoted_match'}
        second_score = scored[1][0] if len(scored) > 1 else 0.0
        if scored[0][0] - second_score < MIN_TOP2_MARGIN:
            return None, {'source': 'slm', 'reason': 'competing_patterns'}

        score, pattern = scored[0]
        try:
            decision = instantiate_template(pattern.template, payload, features, inventory)
        except Exception:
            decision = None
        if decision is None:
            return None, {'source': 'slm', 'reason': 'template_binding_failed'}
        return decision, {
            'source': 'fsm_promoted',
            'pattern_id': pattern.pattern_id,
            'match_score': score,
        }


print('Adaptive FSM/cache definitions loaded.')


## Prequential run

Each row is predicted before its target is used. A trusted FSM decision is attempted first. Otherwise the saved SLM runs. Only a correct SLM decision can reinforce the candidate cache. Existing candidates are tested in shadow on later rows before those rows update the cache.


In [ ]:
#@title Run the adaptive FSM + SLM stream
cache = PatternCache()
router = AdaptiveRouter(cache)
run_logs = []

for step, row in enumerate(stream_rows, 1):
    payload = copy.deepcopy(row['input'])
    inventory = copy.deepcopy(row['inventory'])
    target = row['target']
    started = time.perf_counter()

    decision, route = router.try_fsm(payload, inventory, step)
    raw_output = None
    validation_error = None
    if decision is None:
        route = {'source': 'slm', 'reason': route.get('reason')}
        try:
            raw_output = engine(payload)
            decision = Runtime.validate_decision(
                Runtime.parse_decision(raw_output), payload, inventory
            )
        except Exception as exc:
            decision = None
            validation_error = f'{type(exc).__name__}: {exc}'

    latency_ms = (time.perf_counter() - started) * 1000
    correct = decisions_equal(decision, target)

    # The label is used only after the current row has already been answered.
    cache.shadow_test(payload, inventory, target, step)
    learned_pattern = None
    if route['source'] == 'slm' and correct and decision is not None:
        learned_pattern = cache.learn_confirmed(payload, inventory, decision, step)
    elif route['source'] == 'fsm_promoted':
        pattern = next(p for p in cache.patterns if p.pattern_id == route['pattern_id'])
        if correct:
            pattern.served_successes += 1
            pattern.heat += 1.0
        else:
            cache.demote(pattern, step)

    run_logs.append({
        'step': step,
        'phase': row.get('source_provenance', {}).get('stream_phase', 'unspecified'),
        'scenario': row['scenario'],
        'route': route['source'],
        'reason': route.get('reason'),
        'pattern_id': route.get('pattern_id'),
        'match_score': route.get('match_score'),
        'correct': correct,
        'latency_ms': latency_ms,
        'learned_pattern': learned_pattern,
        'validation_error': validation_error,
        'prediction': decision.model_dump() if decision is not None else None,
        'target': target,
    })

    if step % 10 == 0 or step == len(stream_rows):
        promoted = sum(pattern.status == 'promoted' for pattern in cache.patterns)
        print(f'{step}/{len(stream_rows)} | candidates={len(cache.patterns)} | promoted={promoted}')

print('Run complete.')


In [ ]:
#@title Inspect routing, accuracy, latency, and learned patterns
import pandas as pd
from IPython.display import display

results = pd.DataFrame(run_logs)
summary = {
    'rows': len(results),
    'overall_accuracy': float(results['correct'].mean()) if len(results) else None,
    'fsm_coverage': float(results['route'].str.startswith('fsm').mean()) if len(results) else None,
    'slm_call_rate': float((results['route'] == 'slm').mean()) if len(results) else None,
    'latency_p50_ms': float(results['latency_ms'].median()) if len(results) else None,
    'latency_p95_ms': float(results['latency_ms'].quantile(0.95)) if len(results) else None,
    'candidate_count': len(cache.patterns),
    'promoted_count': sum(pattern.status == 'promoted' for pattern in cache.patterns),
    'promotion_events': cache.promotion_events,
}
print(json.dumps(summary, indent=2))

print('\nBy route')
display(results.groupby('route').agg(
    rows=('step', 'count'),
    accuracy=('correct', 'mean'),
    median_latency_ms=('latency_ms', 'median'),
).reset_index())

print('\nBy stream phase')
display(results.groupby('phase', sort=False).agg(
    rows=('step', 'count'),
    overall_accuracy=('correct', 'mean'),
    fsm_coverage=('route', lambda values: values.str.startswith('fsm').mean()),
    promoted_fsm_coverage=('route', lambda values: (values == 'fsm_promoted').mean()),
    slm_call_rate=('route', lambda values: (values == 'slm').mean()),
    median_latency_ms=('latency_ms', 'median'),
).reset_index())

print('\nBy stream phase and route')
display(results.groupby(['phase', 'route'], sort=False).agg(
    rows=('step', 'count'),
    accuracy=('correct', 'mean'),
    median_latency_ms=('latency_ms', 'median'),
).reset_index())

evaluation_rows = results[results['phase'] == 'evaluation']
evaluation_summary = {
    'rows': len(evaluation_rows),
    'overall_accuracy': float(evaluation_rows['correct'].mean()) if len(evaluation_rows) else None,
    'combined_fsm_accuracy': float(evaluation_rows.loc[evaluation_rows['route'].str.startswith('fsm'), 'correct'].mean()) if evaluation_rows['route'].str.startswith('fsm').any() else None,
    'promoted_fsm_accuracy': float(evaluation_rows.loc[evaluation_rows['route'] == 'fsm_promoted', 'correct'].mean()) if (evaluation_rows['route'] == 'fsm_promoted').any() else None,
    'fallback_slm_accuracy': float(evaluation_rows.loc[evaluation_rows['route'] == 'slm', 'correct'].mean()) if (evaluation_rows['route'] == 'slm').any() else None,
    'fsm_coverage': float(evaluation_rows['route'].str.startswith('fsm').mean()) if len(evaluation_rows) else None,
    'slm_call_rate': float((evaluation_rows['route'] == 'slm').mean()) if len(evaluation_rows) else None,
    'latency_p50_ms': float(evaluation_rows['latency_ms'].median()) if len(evaluation_rows) else None,
    'latency_p95_ms': float(evaluation_rows['latency_ms'].quantile(0.95)) if len(evaluation_rows) else None,
}
print('\nFinal 80-row evaluation phase')
print(json.dumps(evaluation_summary, indent=2))

print('\nBy scenario')
display(results.groupby('scenario').agg(
    rows=('step', 'count'),
    accuracy=('correct', 'mean'),
    fsm_coverage=('route', lambda values: values.str.startswith('fsm').mean()),
).reset_index())

pattern_rows = []
for pattern in cache.patterns:
    pattern_rows.append({
        'pattern_id': pattern.pattern_id,
        'status': pattern.status,
        'action': pattern.template['action'],
        'surfaces': sorted(pattern.surfaces),
        'confirmed': pattern.confirmed,
        'shadow_trials': pattern.shadow_trials,
        'shadow_accuracy': pattern.shadow_accuracy,
        'distinct_examples': len(pattern.examples),
        'distinct_variants': len(pattern.variants),
        'heat': pattern.heat,
        'served_successes': pattern.served_successes,
        'served_failures': pattern.served_failures,
    })

print('\nCandidate cache and promoted FSM patterns')
display(pd.DataFrame(pattern_rows).sort_values(
    ['status', 'heat'], ascending=[False, False]
) if pattern_rows else pd.DataFrame())
